<a href="https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunainakhatwani12/flyrank-ml-internship-sunaina/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install duckdb

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Check that the Colab secret is named "
        "'HF_TOKEN' and notebook access is enabled."
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    );
    """
)

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
MARCH_DATA = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

print("Connection created successfully.")
print("Development window: March 2026")

Connection created successfully.
Development window: March 2026


My lane is Refresh / Content Opportunity Scoring.

The unit of analysis is one pseudonymized content item for one pseudonymized client, summarized over March 2026.

I will use March 2026 as the development window because it is a mid-panel month. I will not use June 2026 for developing the label because June is the final month and should remain a sealed test period.

The main table is `fact_content_daily_performance`. Its daily rows will be aggregated so that the final feature frame contains one row per client and content item for March 2026.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# Inspect the warehouse schema and match the required fields safely.

schema_df = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{MARCH_DATA}')
    """
).df()

warehouse_columns = schema_df["column_name"].tolist()


def choose_column(candidates, required=True):
    """Return the first matching warehouse column name."""
    for candidate in candidates:
        if candidate in warehouse_columns:
            return candidate

    if required:
        raise ValueError(
            "None of these expected columns were found: "
            f"{candidates}\n\nAvailable columns:\n{warehouse_columns}"
        )

    return None


date_col = choose_column(
    ["report_date", "date", "performance_date"]
)

client_col = choose_column(
    ["client_hash_id", "client_id", "client_key"]
)

content_col = choose_column(
    ["content_hash_id", "content_id", "content_key"]
)

impressions_col = choose_column(
    ["gsc_impressions", "impressions", "search_impressions"]
)

clicks_col = choose_column(
    ["gsc_clicks", "clicks", "search_clicks"]
)

position_col = choose_column(
    [
        "gsc_avg_position",
        "gsc_position",
        "avg_position",
        "position",
        "average_position",
    ]
)

sessions_col = choose_column(
    ["ga4_sessions", "sessions", "organic_sessions"],
    required=False
)

availability_col = choose_column(
    [
        "ga4_data_available",
        "gsc_data_available",
        "is_ga4_data_available",
        "ga4_available",
        "has_ga4_data",
    ],
    required=False
)

print("Matched warehouse fields:")
print("Date:", date_col)
print("Client:", client_col)
print("Content:", content_col)
print("Impressions:", impressions_col)
print("Clicks:", clicks_col)
print("Position:", position_col)
print("Sessions:", sessions_col)
print("Availability:", availability_col)

schema_df

Matched warehouse fields:
Date: report_date
Client: client_hash_id
Content: content_hash_id
Impressions: gsc_impressions
Clicks: gsc_clicks
Position: gsc_avg_position
Sessions: ga4_sessions
Availability: ga4_data_available


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


### Feature fields

I use a maximum of five features:

1. `impressions_before` — knowable at the decision moment because it is summed only from March 1–15.
2. `clicks_before` — knowable at the decision moment because it is summed only from March 1–15.
3. `ctr_before` — knowable at the decision moment because it is calculated from earlier clicks and impressions only.
4. `avg_position_before` — knowable at the decision moment because it uses observed search positions from March 1–15 only.
5. `sessions_before` — knowable at the decision moment because it uses available analytics sessions from March 1–15 only.

### Label

`future_decline` is the temporary proxy label. It equals 1 when clicks measured during March 16–31 are lower than clicks measured during March 1–15; otherwise, it equals 0.

### Context fields

`client_hash_id` and `content_hash_id` identify the pseudonymized client and content item. They are retained for grouping and checking the grain, but they are not model features.

`report_date` defines the feature and outcome windows.

### Excluded fields

I exclude URLs, client names, query text, and any other identifying information because the public notebook must remain pseudonymized.

I also exclude `clicks_after`, `future_decline`, and any label-derived field from the honest model because these values would not be known at the decision moment. Including them would cause target leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# QUERY 1: Verify the daily grain
# ============================================================

grain_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT CONCAT(
            CAST({date_col} AS VARCHAR), '|',
            CAST({client_col} AS VARCHAR), '|',
            CAST({content_col} AS VARCHAR)
        )
    ) AS distinct_grain_rows,
    COUNT(*) -
    COUNT(
        DISTINCT CONCAT(
            CAST({date_col} AS VARCHAR), '|',
            CAST({client_col} AS VARCHAR), '|',
            CAST({content_col} AS VARCHAR)
        )
    ) AS duplicate_grain_rows
FROM read_parquet('{MARCH_DATA}')
"""

grain_check = con.execute(grain_query).df()

print("QUERY 1 — GRAIN CHECK")
display(grain_check)


# ============================================================
# QUERY 2: Verify row count and date span
# ============================================================

count_span_query = f"""
SELECT
    COUNT(*) AS slice_rows,
    COUNT(DISTINCT {client_col}) AS pseudonymized_clients,
    COUNT(DISTINCT {content_col}) AS pseudonymized_content_items,
    MIN({date_col}) AS first_date,
    MAX({date_col}) AS last_date
FROM read_parquet('{MARCH_DATA}')
WHERE {date_col} >= DATE '2026-03-01'
  AND {date_col} < DATE '2026-04-01'
"""

count_span_check = con.execute(count_span_query).df()

print("\nQUERY 2 — SLICE SIZE AND DATE SPAN")
display(count_span_check)


# ============================================================
# QUERY 3: Verify availability using IS TRUE
# ============================================================

if availability_col is not None:
    availability_condition = f"{availability_col} IS TRUE"
elif sessions_col is not None:
    # A boolean availability field was not found, so derive one safely
    # and still apply the required IS TRUE syntax.
    availability_condition = (
        f"({sessions_col} IS NOT NULL) IS TRUE"
    )
else:
    # GSC data is present in this table; this fallback checks that
    # the search measurement is available.
    availability_condition = (
        f"({impressions_col} IS NOT NULL) IS TRUE"
    )

availability_query = f"""
SELECT
    COUNT(*) AS rows_before_filter,
    COUNT(*) FILTER (
        WHERE {availability_condition}
    ) AS rows_surviving_is_true,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE {availability_condition}
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS percent_surviving
FROM read_parquet('{MARCH_DATA}')
WHERE {date_col} >= DATE '2026-03-01'
  AND {date_col} < DATE '2026-04-01'
"""

availability_check = con.execute(availability_query).df()

print("\nQUERY 3 — AVAILABILITY CHECK WITH IS TRUE")
display(availability_check)


# ============================================================
# BUILD THE FIVE-FEATURE FRAME
# Decision moment: end of March 15, 2026
# Outcome window: March 16–31, 2026
# ============================================================

sessions_before_sql = (
    f"""
    SUM(
        CASE
            WHEN {date_col} < DATE '2026-03-16'
            THEN COALESCE({sessions_col}, 0)
            ELSE 0
        END
    )
    """
    if sessions_col is not None
    else "CAST(0 AS DOUBLE)"
)

feature_query = f"""
WITH monthly_content AS (
    SELECT
        {client_col} AS client_hash_id,
        {content_col} AS content_hash_id,

        SUM(
            CASE
                WHEN {date_col} < DATE '2026-03-16'
                THEN COALESCE({impressions_col}, 0)
                ELSE 0
            END
        ) AS impressions_before,

        SUM(
            CASE
                WHEN {date_col} < DATE '2026-03-16'
                THEN COALESCE({clicks_col}, 0)
                ELSE 0
            END
        ) AS clicks_before,

        AVG(
            CASE
                WHEN {date_col} < DATE '2026-03-16'
                     AND COALESCE({impressions_col}, 0) > 0
                THEN {position_col}
                ELSE NULL
            END
        ) AS avg_position_before,

        {sessions_before_sql} AS sessions_before,

        SUM(
            CASE
                WHEN {date_col} >= DATE '2026-03-16'
                THEN COALESCE({clicks_col}, 0)
                ELSE 0
            END
        ) AS clicks_after

    FROM read_parquet('{MARCH_DATA}')

    WHERE {date_col} >= DATE '2026-03-01'
      AND {date_col} < DATE '2026-04-01'

    GROUP BY
        {client_col},
        {content_col}
)

SELECT
    client_hash_id,
    content_hash_id,

    impressions_before,
    clicks_before,

    CASE
        WHEN impressions_before > 0
        THEN clicks_before * 1.0 / impressions_before
        ELSE 0
    END AS ctr_before,

    avg_position_before,
    sessions_before,

    clicks_after,

    CASE
        WHEN clicks_after < clicks_before
             AND clicks_before > 0
        THEN 1
        ELSE 0
    END AS future_decline

FROM monthly_content

WHERE impressions_before > 0
"""

feature_frame = con.execute(feature_query).df()

feature_columns = [
    "impressions_before",
    "clicks_before",
    "ctr_before",
    "avg_position_before",
    "sessions_before",
]

print("\nFIVE-FEATURE FRAME")
print("One row = one pseudonymized client-content item.")
print("Feature-frame shape:", feature_frame.shape)
print("Number of features:", len(feature_columns))

display(
    feature_frame[
        [
            "client_hash_id",
            "content_hash_id",
            *feature_columns,
            "future_decline",
        ]
    ].head(10)
)


# ============================================================
# QUICK LEAKAGE EXPERIMENT
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

model_df = feature_frame[
    feature_columns + ["future_decline"]
].replace([np.inf, -np.inf], np.nan).copy()

model_df = model_df.dropna(subset=["future_decline"])

# Use a manageable sample if the frame is very large.
if len(model_df) > 100_000:
    model_df = model_df.sample(
        n=100_000,
        random_state=42
    )

X_honest = model_df[feature_columns]
y = model_df["future_decline"].astype(int)

if y.nunique() < 2:
    raise ValueError(
        "The temporary label contains only one class. "
        "The decline proxy needs adjustment."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=20,
        random_state=42,
    ),
)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)
honest_probabilities = honest_model.predict_proba(X_test)[:, 1]

honest_accuracy = accuracy_score(
    y_test,
    honest_predictions
)

honest_auc = roc_auc_score(
    y_test,
    honest_probabilities
)


# Deliberately create one forbidden label-derived feature.
leaky_model_df = model_df.copy()
leaky_model_df["decline_signal_leak"] = (
    leaky_model_df["future_decline"]
)

leaky_features = feature_columns + ["decline_signal_leak"]

X_leaky = leaky_model_df[leaky_features]

X_leaky_train, X_leaky_test, y_leaky_train, y_leaky_test = (
    train_test_split(
        X_leaky,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y,
    )
)

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=20,
        random_state=42,
    ),
)

leaky_model.fit(
    X_leaky_train,
    y_leaky_train
)

leaky_predictions = leaky_model.predict(
    X_leaky_test
)

leaky_probabilities = leaky_model.predict_proba(
    X_leaky_test
)[:, 1]

leaky_accuracy = accuracy_score(
    y_leaky_test,
    leaky_predictions
)

leaky_auc = roc_auc_score(
    y_leaky_test,
    leaky_probabilities
)

score_comparison = pd.DataFrame(
    {
        "experiment": [
            "Honest five-feature model",
            "Leaky model with label-derived field",
        ],
        "accuracy": [
            honest_accuracy,
            leaky_accuracy,
        ],
        "roc_auc": [
            honest_auc,
            leaky_auc,
        ],
    }
)

print("\nLEAKAGE EXPERIMENT")
display(score_comparison.round(3))

print(
    "The leaky score is artificially high because "
    "`decline_signal_leak` directly contains the answer."
)


# Delete the forbidden field and keep only the honest features.
leaky_model_df = leaky_model_df.drop(
    columns=["decline_signal_leak"]
)

final_feature_columns = feature_columns.copy()

assert "decline_signal_leak" not in leaky_model_df.columns
assert len(final_feature_columns) == 5

print("\nLEAK REMOVED")
print("Final retained features:", final_feature_columns)
print("Honest accuracy kept:", round(honest_accuracy, 3))
print("Honest ROC AUC kept:", round(honest_auc, 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1 — GRAIN CHECK


,total_rows,distinct_grain_rows,duplicate_grain_rows
0,9841378,9841378,0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


QUERY 2 — SLICE SIZE AND DATE SPAN


...


FIVE-FEATURE FRAME
One row = one pseudonymized client-content item.
Feature-frame shape: (151981, 9)
Number of features: 5


,client_hash_id,content_hash_id,impressions_before,clicks_before,ctr_before,avg_position_before,sessions_before,future_decline
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,0.001438,6.327311,0.0,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,0.000000,3.906852,0.0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,0.000810,6.473735,0.0,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,0.003279,7.259861,0.0,1
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,14.0,0.0,0.000000,9.000000,0.0,0
5,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,0.004167,3.860842,0.0,1
6,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,131.0,0.0,0.000000,9.284735,0.0,0
7,client_73cda7b4e4f265ea,content_1f380a642aed423b,44.0,1.0,0.022727,7.872222,0.0,1
8,client_73cda7b4e4f265ea,content_22c063002b7c1caf,172.0,0.0,0.000000,7.602850,0.0,0
9,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,3104.0,16.0,0.005155,5.536679,0.0,1



LEAKAGE EXPERIMENT


,experiment,accuracy,roc_auc
0,Honest five-feature model,0.858,0.927
1,Leaky model with label-derived field,1.000,1.000


The leaky score is artificially high because `decline_signal_leak` directly contains the answer.

LEAK REMOVED
Final retained features: ['impressions_before', 'clicks_before', 'ctr_before', 'avg_position_before', 'sessions_before']
Honest accuracy kept: 0.858
Honest ROC AUC kept: 0.927


I verify the contract with exactly three small checks.

**Query 1 — grain:** I check whether the March table contains duplicate combinations of report date, pseudonymized client, and pseudonymized content item.

**Query 2 — size and date span:** I count the rows and distinct content items in my March slice and display its minimum and maximum dates.

**Query 3 — availability:** I apply an explicit `IS TRUE` availability filter and count how many rows survive.

After the three checks, I aggregate the first half of March into a five-feature frame. I use the second half only to create the temporary future-decline proxy.

Finally, I deliberately add one label-derived field to demonstrate leakage. Its score should jump toward perfect. I then remove it and retain the honest score.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# Summarize limitations that are measurable in this March slice.

limits_summary = pd.DataFrame(
    {
        "check": [
            "March rows",
            "Distinct pseudonymized clients",
            "Distinct pseudonymized content items",
            "Rows surviving availability check",
            "Feature-frame rows",
            "Rows with missing average position",
            "Temporary decline-label rate",
        ],
        "value": [
            int(count_span_check.loc[0, "slice_rows"]),
            int(count_span_check.loc[0, "pseudonymized_clients"]),
            int(count_span_check.loc[0, "pseudonymized_content_items"]),
            int(availability_check.loc[0, "rows_surviving_is_true"]),
            int(len(feature_frame)),
            int(feature_frame["avg_position_before"].isna().sum()),
            round(float(feature_frame["future_decline"].mean()), 3),
        ],
    }
)

display(limits_summary)

print(
    "\nNamed limitation: client histories and analytics availability "
    "are uneven, so missing or shorter histories must not be treated "
    "as identical to true zero performance."
)

,check,value
0,March rows,9841378.000
1,Distinct pseudonymized clients,55.000
2,Distinct pseudonymized content items,331437.000
3,Rows surviving availability check,413966.000
4,Feature-frame rows,151981.000
5,Rows with missing average position,0.000
6,Temporary decline-label rate,0.191



Named limitation: client histories and analytics availability are uneven, so missing or shorter histories must not be treated as identical to true zero performance.


This slice has several important limitations.

First, the warehouse is an **unbalanced panel**. Different pseudonymized clients have different search and analytics history depths, so not every client contributes the same number of observed days.

Second, some early rows may contain Search Console measurements but not GA4 measurements. Missing sessions do not necessarily mean zero real user activity; they may mean analytics data was unavailable.

Third, this exercise uses two halves of one month. The feature and outcome windows do not overlap, but they are close together and may be affected by short-term volatility, seasonality, reporting delay, or one unusual event.

Fourth, the temporary label only measures whether later clicks are lower than earlier clicks. It cannot prove why performance changed, whether the content was edited, whether rankings changed because of competition, or whether an editor should definitely refresh the page.

Finally, this observational dataset cannot establish causation. The model output must remain measured, directional decision-support and should be reviewed by a human editor.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.